# TP1 - Compteur de mots dans les fichiers txt

L'objectif de ce travail est de mettre en œuvre trois approches visant à compter le nombre d'occurrences de chaque mot dans un fichier .txt, tout en comprenant le fonctionnement de chacune d'elles.

Les 3 versions proposées sont :

- Version simple utilisant Python

- Pseudo-MapReduce

- Hadoop MapReduce

Pour évaluer chaque approche, le temps d'exécution de chacune sera calculé en utilisant des fichiers texte contenant des nombres de mots différents.

## Importation des bibliothèques utilisées

In [11]:
from pathlib import Path
import random
from time import perf_counter
import pandas as pd
from collections import defaultdict

## Création des fichiers txt

Quatre fichiers .txt contenant des mots aléatoires sur chaque ligne ont été créés. Chaque fichier contient dix fois plus de mots que le précédent, dans le but d'évaluer les performances de chaque approche pour différentes tailles de fichiers d'entrée.

In [ ]:
n = 1000
tailles = [1, 10, 100, 1000]
mots = ["chat", "chien", "arbre", "fleur", "maison", "soleil", "montagne", "livre", "ciel", "mer", "lune", "route", "oiseau", "jardin", "voiture"]

dossier = Path("data")
dossier.mkdir(exist_ok=True)

fichiers = {}

for m in tailles:
    lignes = n * m
    chemin = dossier / f"mots_taille_{m}.txt"

    with chemin.open("w", encoding="utf-8") as fichier:
        for i in range(lignes):
            mot = random.choice(mots)
            fichier.write(mot + "\n")

    fichiers[m] = chemin

    print(f"{chemin.name} : {lignes} lignes, {chemin.stat().st_size} bytes")

mots_taille_1.txt : 1000 lignes, 6309 bytes
mots_taille_10.txt : 10000 lignes, 62711 bytes
mots_taille_100.txt : 100000 lignes, 626776 bytes
mots_taille_1000.txt : 1000000 lignes, 6265995 bytes


## Version simple utilisant Python

In [9]:
def compter_occurrences(fichier):
    with open(fichier, 'r', encoding = 'utf-8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur [mot] = compteur.get(mot, 0) + 1
    return compteur

if __name__ == "__main__":
    dossier = Path("data")
    fichiers = sorted(dossier.glob("*.txt"))
    for fichier in fichiers:
        start = perf_counter()
        resultats = compter_occurrences(fichier)
        end = perf_counter()
        temps = end - start
        print(f"fichier : {fichier.name}\ntemps : {temps:.6f} secondes\nnombre de mots : {sum(resultats.values())}\nnombre de mots uniques : {len(resultats)}\n\nOccurrences :")

        for mot, nb in sorted(resultats.items(), key = lambda x : x [1], reverse = True):
            print (f"{mot} : {nb}")
        print("\n\n\n")

fichier : mots_taille_1.txt
temps : 0.000249 secondes
nombre de mots : 1000
nombre de mots uniques : 15

Occurrences :
voiture : 82
lune : 78
livre : 75
montagne : 73
arbre : 73
oiseau : 70
maison : 68
chat : 68
jardin : 66
route : 65
fleur : 61
mer : 59
ciel : 59
chien : 58
soleil : 45




fichier : mots_taille_10.txt
temps : 0.003138 secondes
nombre de mots : 10000
nombre de mots uniques : 15

Occurrences :
maison : 698
lune : 694
soleil : 688
route : 683
arbre : 676
chien : 667
montagne : 666
livre : 664
jardin : 663
voiture : 661
mer : 655
chat : 653
fleur : 651
ciel : 641
oiseau : 640




fichier : mots_taille_100.txt
temps : 0.013275 secondes
nombre de mots : 100000
nombre de mots uniques : 15

Occurrences :
chien : 6781
oiseau : 6757
livre : 6746
montagne : 6725
fleur : 6709
arbre : 6689
lune : 6675
chat : 6672
ciel : 6668
maison : 6639
soleil : 6630
route : 6616
jardin : 6592
mer : 6551
voiture : 6550




fichier : mots_taille_1000.txt
temps : 0.114798 secondes
nombre de mots :

## Pseudo-MapReduce

In [10]:
def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced [mot] = sum(valeurs)
    return reduced

if __name__ == "__main__":
    dossier = Path("data")
    fichiers = sorted(dossier.glob("*.txt"))
    for fichier in fichiers:
        mapped = []
        start = perf_counter()
        with open(fichier, "r", encoding = "utf-8") as f:
            for ligne in f:
                mapped.extend(mapper(ligne))

        grouped = shuffle(mapped)
        reduced = reducer(grouped)

        end = perf_counter()
        temps = end - start
        print(f"fichier : {fichier.name}\ntemps : {temps:.6f} secondes\nnombre de mots : {sum(reduced.values())}\nnombre de mots uniques : {len(reduced)}\n\nOccurrences :")

        for mot, count in sorted(reduced.items(), key = lambda x : x[1] , reverse = True):
            print (f"{mot} : {count}")
        print("\n\n\n")

fichier : mots_taille_1.txt
temps : 0.001191 secondes
nombre de mots : 1000
nombre de mots uniques : 15

Occurrences :
voiture : 82
lune : 78
livre : 75
montagne : 73
arbre : 73
oiseau : 70
maison : 68
chat : 68
jardin : 66
route : 65
fleur : 61
mer : 59
ciel : 59
chien : 58
soleil : 45




fichier : mots_taille_10.txt
temps : 0.006758 secondes
nombre de mots : 10000
nombre de mots uniques : 15

Occurrences :
maison : 698
lune : 694
soleil : 688
route : 683
arbre : 676
chien : 667
montagne : 666
livre : 664
jardin : 663
voiture : 661
mer : 655
chat : 653
fleur : 651
ciel : 641
oiseau : 640




fichier : mots_taille_100.txt
temps : 0.040867 secondes
nombre de mots : 100000
nombre de mots uniques : 15

Occurrences :
chien : 6781
oiseau : 6757
livre : 6746
montagne : 6725
fleur : 6709
arbre : 6689
lune : 6675
chat : 6672
ciel : 6668
maison : 6639
soleil : 6630
route : 6616
jardin : 6592
mer : 6551
voiture : 6550




fichier : mots_taille_1000.txt
temps : 0.386150 secondes
nombre de mots :

## Hadoop MapReduce

#### fichier : mots_taille_1.txt

#### temps : 13.03 secondes

#### nombre de mots : 1000

#### nombre de mots uniques : 15

#### Occurrences :

voiture : 82

lune : 78

livre : 75

arbre : 73

montagne : 73

oiseau : 70

chat : 68

maison : 68

jardin : 66

route : 65

fleur : 61

ciel : 59

mer : 59

chien : 58

soleil : 45

#### fichier : mots_taille_10.txt

#### temps : 14.08 secondes

#### nombre de mots : 10000

#### nombre de mots uniques : 15

#### Occurrences :

maison : 698

lune : 694

soleil : 688

route : 683

arbre : 676

chien : 667

montagne : 666

livre : 664

jardin : 663

voiture :661

mer :655

chat : 653

fleur : 651

ciel : 641

oiseau : 640


#### fichier : mots_taille_100.txt

#### temps : 14.29 secondes

#### nombre de mots : 100000

#### nombre de mots uniques : 15

#### Occurrences :

chien :6781

oiseau :6757

livre :6746

montagne :6725

fleur : 6709

arbre : 6689

lune : 6675

chat : 6672

ciel : 6668

maison : 6639

soleil : 6630

route : 6616

jardin : 6592

mer : 6551

voiture : 6550




#### fichier : mots_taille_1000.txt

#### temps : 15.01 secondes

#### nombre de mots : 1000000

#### nombre de mots uniques : 15


#### Occurrences :

route : 66951

chat : 66935

maison : 66885

jardin : 66882

arbre : 66821

mer : 66745

chien : 66699

fleur : 66698

ciel : 66691

oiseau : 66682

voiture : 66534

soleil :66527

montagne : 66437

livre : 66269

lune : 66244

## Temps d'exécution pour chaque approche sur chaque fichier testé (en secondes)

In [19]:
resultats = {
    "Fichier": ["mots_taille_1.txt", "mots_taille_10.txt", "mots_taille_100.txt", "mots_taille_1000.txt"],
    "Python simple": [0.000249, 0.003138, 0.013275, 0.114798],
    "Pseudo-MapReduce": [0.001191, 0.006758, 0.040867, 0.386150],
    "Hadoop MapReduce": [13.03, 14.08, 14.29, 15.01],
}

df = pd.DataFrame(resultats)

display(df)

,Fichier,Python simple,Pseudo-MapReduce,Hadoop MapReduce
0,mots_taille_1.txt,0.000249,0.001191,13.03
1,mots_taille_10.txt,0.003138,0.006758,14.08
2,mots_taille_100.txt,0.013275,0.040867,14.29
3,mots_taille_1000.txt,0.114798,0.386150,15.01


## Analyse des performances et conclusion

L'approche simple utilisant Python est la méthode la plus rapide parmi celles présentées pour les données utilisées. Le pseudo-MapReduce présente un temps d'exécution légèrement plus long en raison de l'utilisation des étapes de Map, Shuffle et Reduce. Hadoop MapReduce est considéré comme la solution la plus stable pour des volumes de données extrêmement importants, mais il implique un temps d'exécution de base élevé dû à la complexité de l'infrastructure.

Ce TP a permis de comprendre trois approches de complexités différentes. Le choix de la méthode dépend directement de la taille de l'ensemble de données utilisé. Les petits jeux de données conviennent mieux au pseudo-MapReduce et, surtout, à l'approche simple utilisant Python, tandis que le traitement de Big Data justifie le recours à Hadoop MapReduce.